In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.report_payment_migration AS
SELECT
    c.Customer_ID,

    -- HSIC
    SUM(CASE WHEN m.Fiscal_Year = 2025 THEN m.HSIC_Spend ELSE 0 END) AS FY2025_HSIC,
    SUM(CASE WHEN m.Fiscal_Year = 2026 THEN m.HSIC_Spend ELSE 0 END) AS FY2026_HSIC,

    -- Competitor Card
    SUM(CASE WHEN m.Fiscal_Year = 2025 THEN m.Competitor_Card_Spend ELSE 0 END) AS FY2025_Competitor_Card,
    SUM(CASE WHEN m.Fiscal_Year = 2026 THEN m.Competitor_Card_Spend ELSE 0 END) AS FY2026_Competitor_Card,

    -- Debit
    SUM(CASE WHEN m.Fiscal_Year = 2025 THEN m.Debit_Card_Spend ELSE 0 END) AS FY2025_Debit,
    SUM(CASE WHEN m.Fiscal_Year = 2026 THEN m.Debit_Card_Spend ELSE 0 END) AS FY2026_Debit,

    -- Cash / UPI
    SUM(CASE WHEN m.Fiscal_Year = 2025 THEN m.Cash_UPI_Spend ELSE 0 END) AS FY2025_Cash_UPI,
    SUM(CASE WHEN m.Fiscal_Year = 2026 THEN m.Cash_UPI_Spend ELSE 0 END) AS FY2026_Cash_UPI,

    -- Wallet
    SUM(CASE WHEN m.Fiscal_Year = 2025 THEN m.Wallet_Spend ELSE 0 END) AS FY2025_Wallet,
    SUM(CASE WHEN m.Fiscal_Year = 2026 THEN m.Wallet_Spend ELSE 0 END) AS FY2026_Wallet

FROM synchrony.analytics.customer_movement_analysis c
JOIN synchrony.analytics.customer_monthly_sow m
    ON c.Customer_ID = m.Customer_ID

WHERE c.SoW_Movement = 'SoW_DECLINED'
  AND c.HSIC_Movement = 'HSIC_SPEND_DOWN'
  AND c.Total_Spend_Movement = 'TOTAL_SPEND_UP'

GROUP BY c.Customer_ID;

In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.report_payment_migration AS
SELECT
    *,
    
    FY2026_HSIC - FY2025_HSIC AS HSIC_Change,
    FY2026_Competitor_Card - FY2025_Competitor_Card AS Competitor_Card_Change,
    FY2026_Debit - FY2025_Debit AS Debit_Change,
    FY2026_Cash_UPI - FY2025_Cash_UPI AS Cash_UPI_Change,
    FY2026_Wallet - FY2025_Wallet AS Wallet_Change

FROM synchrony.analytics.report_payment_migration;

In [0]:
%sql
SELECT
    COUNT(*) AS Customer_Count,
    SUM(FY2025_HSIC) AS FY2025_HSIC,
    SUM(FY2026_HSIC) AS FY2026_HSIC,
    SUM(HSIC_Change) AS HSIC_Change,
    SUM(Competitor_Card_Change) AS Competitor_Card_Change,
    SUM(Debit_Change) AS Debit_Change,
    SUM(Cash_UPI_Change) AS Cash_UPI_Change,
    SUM(Wallet_Change) AS Wallet_Change
FROM synchrony.analytics.report_payment_migration;